# EasyRAG Normalization And Cleaning

## Chapter position

This chapter sits before indexing. Raw files or in-memory text become canonical `Document` objects here, so mistakes at this layer tend to echo through every later stage.

## Learning question

What exactly changes during normalization, and how do you tell whether cleaning improved the text instead of damaging it?

## Success criteria

- inspect `normalize_document_text()` on messy inputs
- compare raw text, normalized text, and prepared `Document` metadata
- see how normalization decisions affect what later stages receive

## Source code anchors

- `easyrag.rag.indexing.normalization.normalize_document_text`
- `easyrag.rag.orchestrator.EasyRAG.prepare_documents`
- `easyrag.rag.indexing.prepare.prepare_documents_for_insert_with_report`


## Method principles

- `easyrag.rag.indexing.normalization.normalize_document_text`: This normalization helper cleans line endings, null bytes, trailing whitespace, and excessive blank lines while keeping markdown-like structure readable. It is intentionally small and predictable.
- `easyrag.rag.orchestrator.EasyRAG.prepare_documents`: This public helper wraps the document-preparation layer. It turns raw text plus optional IDs and logical paths into canonical `Document` objects before any index write happens.
- `easyrag.rag.indexing.prepare.prepare_documents_for_insert_with_report`: This is the same preparation path plus a small report about skipped empty inputs. It is useful when you want to see what normalization removed before insertion begins.


## How the code fits together

The flow in this notebook is raw text -> normalized text -> preparation report -> insertion boundary. The goal is not to show every internal detail at once. The goal is to keep the boundary for this stage visible enough that later behavior still feels explainable. If a result looks odd, the intermediate objects in this notebook should tell you where to look next.

## What this cell is proving

We start by making the repo importable from the notebook runtime and loading the helpers this notebook depends on. This cell should stay quiet. It is only here to make the later examples reliable.

In [ ]:
import importlib
import json
import sys
from pathlib import Path

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "notebooks" / "_utils.py").exists():
        REPO_ROOT = candidate.resolve()
        if str(REPO_ROOT) not in sys.path:
            sys.path.insert(0, str(REPO_ROOT))
        break
else:
    raise RuntimeError("Could not locate the EasyRAG repository root.")

EasyRAG = importlib.import_module("easyrag.rag").EasyRAG
normalization = importlib.import_module("easyrag.rag.indexing.normalization")
prepare = importlib.import_module("easyrag.rag.indexing.prepare")
providers = importlib.import_module("easyrag.rag.providers")
normalize_document_text = normalization.normalize_document_text
prepare_documents_for_insert_with_report = (
    prepare.prepare_documents_for_insert_with_report
)
can_use_openai_compatible_models = providers.can_use_openai_compatible_models


def show_case(label: str, raw_text: str, normalized_text: str) -> None:
    print(f"--- {label} ---")
    print("RAW")
    print(repr(raw_text))
    print("NORMALIZED")
    print(repr(normalized_text))
    print()

## What this cell is proving

This path is fully local. It uses the normalization helpers directly and does not require a model provider.

In [ ]:
raw_texts = [
    "# Intro\r\n\r\nEasyRAG uses query rewrite.\x00\n\n## Retrieval\nHybrid retrieval stays inspectable.   \n",
    "   \x00\n\n\n",
    "Title\n\n\nLine A\r\nLine B\r\n\r\nLine C   ",
]
file_paths = ["docs/intro.md", "docs/blank.md", "notes/cleaning.txt"]
doc_ids = ["doc::intro", "doc::blank", "doc::cleaning"]

normalized_texts = [normalize_document_text(text) for text in raw_texts]
for index, (raw_text, normalized_text) in enumerate(
    zip(raw_texts, normalized_texts, strict=False), start=1
):
    show_case(f"sample {index}", raw_text, normalized_text)

prepared_documents = EasyRAG.prepare_documents(
    raw_texts, ids=doc_ids, file_paths=file_paths
)
prepared_with_report, preparation_report = prepare_documents_for_insert_with_report(
    raw_texts, ids=doc_ids, file_paths=file_paths
)

print("Prepared document metadata")
for document in prepared_documents:
    print(document.page_content)
    print(json.dumps(document.metadata, indent=2, sort_keys=True))
    print()

print("Preparation report")
print(json.dumps(preparation_report, indent=2, sort_keys=True))
print(
    f"Prepared counts agree: {len(prepared_documents)} == {len(prepared_with_report)}"
)

## Why this output looks like this

Notice what normalization does and does not do. The helper removes null bytes, standardizes line endings, trims trailing whitespace, and collapses repeated blank lines. It does not rewrite the prose for you. That conservative behavior is useful because the next stages still need the original structure and wording.

In [ ]:
normalization_summary = []
for raw_text, normalized_text, path in zip(
    raw_texts, normalized_texts, file_paths, strict=False
):
    normalization_summary.append(
        {
            "path": path,
            "raw_chars": len(raw_text),
            "normalized_chars": len(normalized_text),
            "changed": raw_text != normalized_text,
            "is_empty_after_normalization": not bool(normalized_text.strip()),
        }
    )

print(json.dumps(normalization_summary, indent=2, sort_keys=True))

## What this cell is proving

The provider-backed path should preserve the same contract even when the backend behavior is less deterministic.

Normalization itself is provider-independent, but the next cell shows that the same cleaned inputs can be inserted into a provider-backed `EasyRAG` instance when the environment is configured.

In [ ]:
if not can_use_openai_compatible_models():
    print("Skipping provider-backed path because OPENAI-compatible config is not set.")
else:
    import tempfile
    from easyrag.rag import QueryParam
    from easyrag.support.async_utils import run_sync

    provider_tmp = tempfile.TemporaryDirectory()
    provider_rag = EasyRAG(
        working_dir=provider_tmp.name, workspace="provider-normalization-demo"
    )
    try:
        run_sync(provider_rag.initialize_storages())
        stats = run_sync(
            provider_rag.ainsert(raw_texts, ids=doc_ids, file_paths=file_paths)
        )
        result = run_sync(
            provider_rag.aquery(
                "How does EasyRAG use query rewrite?",
                QueryParam(mode="hybrid", chunk_top_k=2),
            )
        )
        print(json.dumps(stats, indent=2, sort_keys=True))
        print(json.dumps(result.metadata, indent=2, sort_keys=True, default=str))
    except Exception as exc:
        print(f"Provider-backed path was skipped due to runtime error: {exc}")
    finally:
        try:
            run_sync(provider_rag.finalize_storages())
        finally:
            provider_tmp.cleanup()

## Common mistakes / debugging cues

- If a later stage looks wrong, inspect `doc_id`, logical path, title, and normalized text here first.
- Do not confuse canonical `Document` preparation with actual index writes. They are different boundaries.
- Noisy or empty documents usually create problems long before retrieval. Loading and quality checks are where you catch them cheaply.

## Takeaway

Two things changed between the raw inputs and the prepared documents: the text became canonical, and the metadata became stable. The cleaned text protects later stages from trivial formatting noise, while the derived `path`, `relative_path`, `title`, and `doc_id` fields give retrieval and citations a stable identity.

## Where to go next

- Continue with [02_05_document_quality_and_edge_cases.ipynb](02_05_document_quality_and_edge_cases.ipynb) to see which ingestion problems survive normalization and still need explicit quality flags.
- Read [02-data-loading-overview.md](../../docs/02-data-loading-overview.md) for the chapter-level explanation of loading, cleaning, and normalization.
- Compare this notebook with [02_02_manual_document_preparation.ipynb](02_02_manual_document_preparation.ipynb) if your documents start in memory instead of on disk.